# Panel de viaje · Fase 1.4 · Puntos de interés por pueblo

**Proyecto:** https://github.com/tragabytes/panel-viaje  
**Sesión:** 05 — evaluación de Wikidata SPARQL, Wikipedia API y Overpass como fuentes de POIs por municipio

## Qué evalúa este notebook

1. **Wikidata SPARQL** como fuente estructurada de POIs (monumentos + naturaleza) por municipio.
2. **Wikipedia REST API** para extraer resúmenes cortos y miniaturas de los POIs.
3. **Overpass API** como fallback geográfico cuando Wikidata no devuelve suficiente.
4. **Wikidata SPARQL** (segunda consulta) para datos del municipio: población, altitud, comarca, superficie. Esto cierra de paso el bloque 1.5 del plan.
5. **Overpass** para encontrar los pueblos más cercanos a un punto GPS. Crítico: alimenta la vista 3 entera.

## Muestra de pueblos

| Municipio | Provincia | Habitantes aprox | Por qué está en la muestra |
|---|---|---|---|
| Medinaceli | Soria | ~700 | Pueblo pequeño con historia romana. Caso del plan. Esperable: arco romano, castillo, colegiata. |
| Albarracín | Teruel | ~1000 | Pueblo pequeño con mucho turismo. Esperable: cobertura alta. |
| Chinchón | Madrid | ~5000 | Mediano con patrimonio reconocible (plaza mayor, castillo). |
| Trujillo | Cáceres | ~9000 | Mediano con mucho patrimonio (castillo, Pizarro, iglesias). |
| Castrillo de los Polvazares | León | ~100 | Caso límite: muy pequeño. Aquí esperamos que Wikidata falle y tengamos que recurrir a Overpass. |

## Tipos de POI que buscamos

Patrimonio histórico + naturaleza: castillos, iglesias, monasterios, palacios, puentes históricos, yacimientos arqueológicos, arcos romanos, murallas, torres, miradores, picos, espacios naturales. Sin museos, sin plazas, sin bares.

## Cómo ejecutar

- Abre este notebook con el badge `Open in Colab` del README.
- Ejecuta las celdas de arriba abajo. Cada prueba es independiente.
- Al final hay un resumen que se puede copiar al documento de seguimiento.

## Setup: imports, constantes y helpers

In [ ]:
import requests
import json
import time
from urllib.parse import quote

# User-Agent propio, obligatorio para Wikidata/Wikipedia y buena ciudadanía en Overpass.
# Quedó como estándar del proyecto desde la sesión 03 (problema P02 con Photon).
UA = "panel-viaje-tragabytes/0.1 (https://github.com/tragabytes/panel-viaje; contacto via GitHub)"
HEADERS = {"User-Agent": UA, "Accept": "application/json"}

# Endpoints
WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"
WIKIPEDIA_REST_ES = "https://es.wikipedia.org/api/rest_v1"
OVERPASS = "https://overpass-api.de/api/interpreter"

# Los 5 municipios de la muestra con su Q de Wikidata.
# Verificados manualmente en wikidata.org antes de escribir el notebook.
# El Q es el del municipio, no el de la villa histórica (en casos donde hay ambos).
MUNICIPIOS = [
    {"nombre": "Medinaceli",                   "qid": "Q1026262", "lat": 41.1731, "lon": -2.4344, "habs_aprox": 700,  "caso": "pequeño con historia"},
    {"nombre": "Albarracín",                   "qid": "Q15088",   "lat": 40.4064, "lon": -1.4433, "habs_aprox": 1000, "caso": "pequeño turístico"},
    {"nombre": "Chinchón",                     "qid": "Q494083",  "lat": 40.1396, "lon": -3.4261, "habs_aprox": 5000, "caso": "mediano"},
    {"nombre": "Trujillo",                     "qid": "Q255245",  "lat": 39.4600, "lon": -5.8825, "habs_aprox": 9000, "caso": "mediano con mucho patrimonio"},
    {"nombre": "Castrillo de los Polvazares",  "qid": "Q1053486", "lat": 42.4603, "lon": -6.1364, "habs_aprox": 100,  "caso": "muy pequeño, caso límite"},
]

# Patrón de reintentos estándar del proyecto (validado en sesión 04 con Open-Meteo).
# Timeout 30 s, hasta 3 intentos, esperas crecientes 3 s y 8 s.
def fetch_json(url, method="GET", data=None, headers=None, max_intentos=3, timeout=30):
    hdrs = headers or HEADERS
    esperas = [0, 3, 8]
    last_err = None
    for intento in range(max_intentos):
        if esperas[intento] > 0:
            time.sleep(esperas[intento])
        t0 = time.time()
        try:
            if method == "POST":
                r = requests.post(url, data=data, headers=hdrs, timeout=timeout)
            else:
                r = requests.get(url, headers=hdrs, timeout=timeout)
            dt = (time.time() - t0) * 1000
            if r.status_code == 200:
                return {"ok": True, "data": r.json(), "ms": dt, "bytes": len(r.content), "intentos": intento + 1}
            last_err = f"HTTP {r.status_code}: {r.text[:200]}"
        except Exception as e:
            last_err = f"{type(e).__name__}: {e}"
    return {"ok": False, "error": last_err, "intentos": max_intentos}

print("Setup OK")
print(f"Muestra: {len(MUNICIPIOS)} municipios")
for m in MUNICIPIOS:
    print(f"  {m['nombre']:32s} {m['qid']:10s}  ({m['caso']})")

## Prueba 1 — Wikidata SPARQL: POIs por municipio

**Estrategia:** para cada municipio, lanzamos una consulta SPARQL que busca elementos `wdt:P131*` (located in administrative entity, transitivamente) dentro del Q del municipio, filtrados por instancia de una lista amplia de clases de POI (castillos, iglesias, monumentos, picos, etc.).

**Qué medimos:**
- Número total de POIs devueltos por municipio.
- Cuántos tienen imagen (`P18`).
- Cuántos tienen artículo en Wikipedia en español.
- Latencia por consulta.

**Riesgo conocido:** Wikidata depende de curación humana. Pueblos pequeños pueden no tener los POIs enlazados con `P131` hacia el municipio, o no tener los POIs en absoluto. Si sale poco para Medinaceli o Albarracín, hay un problema de estrategia. Si solo sale poco para Castrillo, es lo esperado.

In [ ]:
# Clases de Wikidata que consideramos POIs válidos para el panel.
# Seleccionadas para cubrir patrimonio histórico + naturaleza, excluyendo museos/plazas/bares.
CLASES_POI = [
    ("Q23413",    "castillo"),
    ("Q16970",    "iglesia"),
    ("Q2977",     "catedral"),
    ("Q108325",   "capilla"),
    ("Q44613",    "monasterio"),
    ("Q317557",   "iglesia parroquial"),
    ("Q16560",    "palacio"),
    ("Q12518",    "torre"),
    ("Q57821",    "fortificación"),
    ("Q839954",   "yacimiento arqueológico"),
    ("Q570116",   "atracción turística"),
    ("Q2319498",  "atalaya"),
    ("Q1440300",  "monumento histórico"),
    ("Q4989906",  "monumento"),
    ("Q1549591",  "edificio histórico"),
    ("Q12280",    "puente"),
    ("Q1486652",  "puente de piedra"),
    ("Q1189170",  "arco triunfal"),
    ("Q2425052",  "puerta de la ciudad"),
    ("Q2221906",  "lugar geográfico"),
    ("Q8502",     "montaña"),
    ("Q207326",   "mirador"),
    ("Q46831",    "cordillera"),
    ("Q4022",     "río"),
    ("Q34038",    "cascada"),
    ("Q23397",    "lago"),
    ("Q35509",    "cueva"),
    ("Q473972",   "área protegida"),
    ("Q46169",    "monumento natural"),
]

def construir_query_pois(qid_municipio):
    """Consulta SPARQL: POIs contenidos (transitivamente) en el municipio,
    filtrados por instancia de las clases que consideramos válidas.
    Usa wdt:P31/wdt:P279* para capturar también subclases."""
    valores = " ".join(f"wd:{q}" for q, _ in CLASES_POI)
    return f'''
SELECT DISTINCT ?item ?itemLabel ?itemDescription ?instanceLabel ?image ?coord ?article WHERE {{
  ?item wdt:P131* wd:{qid_municipio} .
  ?item wdt:P31 ?instance .
  ?instance wdt:P279* ?parentClass .
  VALUES ?parentClass {{ {valores} }}
  OPTIONAL {{ ?item wdt:P18 ?image. }}
  OPTIONAL {{ ?item wdt:P625 ?coord. }}
  OPTIONAL {{
    ?article schema:about ?item ;
             schema:isPartOf <https://es.wikipedia.org/> .
  }}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "es,en". }}
}}
LIMIT 50
'''

def consultar_wikidata(query):
    url = f"{WIKIDATA_SPARQL}?format=json&query={quote(query)}"
    hdrs = {"User-Agent": UA, "Accept": "application/sparql-results+json"}
    return fetch_json(url, headers=hdrs)

# Lanzamos la consulta para cada municipio de la muestra.
resultados_wd = {}

for m in MUNICIPIOS:
    print(f"\n=== {m['nombre']} ({m['qid']}) ===")
    q = construir_query_pois(m['qid'])
    res = consultar_wikidata(q)
    if not res['ok']:
        print(f"  ERROR: {res['error']}")
        resultados_wd[m['nombre']] = {"ok": False, "error": res['error'], "pois": []}
        continue
    rows = res['data']['results']['bindings']
    # Deduplicamos por item URI (la consulta puede devolver varios rows por mismo item si tiene varias P31)
    pois_dict = {}
    for row in rows:
        uri = row['item']['value']
        if uri not in pois_dict:
            pois_dict[uri] = {
                "qid": uri.rsplit('/', 1)[-1],
                "label": row.get('itemLabel', {}).get('value', ''),
                "desc": row.get('itemDescription', {}).get('value', ''),
                "instances": set(),
                "image": row.get('image', {}).get('value'),
                "coord": row.get('coord', {}).get('value'),
                "article": row.get('article', {}).get('value'),
            }
        pois_dict[uri]["instances"].add(row.get('instanceLabel', {}).get('value', ''))
    pois = list(pois_dict.values())
    con_imagen = sum(1 for p in pois if p['image'])
    con_articulo = sum(1 for p in pois if p['article'])
    print(f"  {len(pois)} POIs · {con_imagen} con imagen · {con_articulo} con artículo es.wiki · {res['ms']:.0f} ms · {res['bytes']} B")
    for p in pois[:10]:
        img = "📷" if p['image'] else "  "
        art = "📄" if p['article'] else "  "
        inst = ", ".join(sorted(i for i in p['instances'] if i))[:40]
        print(f"    {img}{art} {p['label']:35s}  ({inst})")
    if len(pois) > 10:
        print(f"    ... y {len(pois)-10} más")
    resultados_wd[m['nombre']] = {
        "ok": True,
        "ms": res['ms'],
        "bytes": res['bytes'],
        "pois": pois,
        "con_imagen": con_imagen,
        "con_articulo": con_articulo,
    }
    time.sleep(1)  # cortesía con Wikidata

print("\n--- Resumen prueba 1 ---")
print(f"{'Municipio':32s} {'POIs':>6s} {'Img':>6s} {'Wiki':>6s} {'ms':>8s}")
for m in MUNICIPIOS:
    r = resultados_wd.get(m['nombre'], {})
    if r.get('ok'):
        print(f"{m['nombre']:32s} {len(r['pois']):>6d} {r['con_imagen']:>6d} {r['con_articulo']:>6d} {r['ms']:>8.0f}")
    else:
        print(f"{m['nombre']:32s}   ERROR: {r.get('error','?')[:40]}")

## Prueba 2 — Wikipedia REST API: resúmenes y miniaturas

**Estrategia:** para los POIs devueltos por Wikidata que tienen artículo en Wikipedia en español, llamamos al endpoint `GET /page/summary/{title}` de la Wikipedia REST API. Este endpoint devuelve:
- `extract`: las primeras frases del artículo, ideal para descripción corta.
- `thumbnail.source`: URL directa a una miniatura del artículo (tamaño por defecto ~320 px).
- `originalimage.source`: URL a la imagen en tamaño original (lo usaremos sólo para saber si existe).

**Qué medimos:**
- Latencia del endpoint.
- Tamaño del JSON.
- Longitud del `extract` (queremos recortar a 1-2 frases).
- **Tamaño real en KB de la miniatura** descargándola.
- Verificar que las URLs de Wikimedia son directas y estables.

**Qué POIs probamos:** hasta 3 por municipio, los que tengan artículo. Esto nos da una muestra razonable sin saturar la API.

In [ ]:
def titulo_desde_url(article_url):
    """https://es.wikipedia.org/wiki/Castillo_de_Medinaceli -> Castillo_de_Medinaceli"""
    return article_url.rsplit('/wiki/', 1)[-1]

def wiki_summary(titulo):
    url = f"{WIKIPEDIA_REST_ES}/page/summary/{titulo}"
    return fetch_json(url)

def tamaño_miniatura(thumb_url):
    """Descarga la miniatura y devuelve su tamaño en bytes."""
    try:
        r = requests.get(thumb_url, headers={"User-Agent": UA}, timeout=15)
        if r.status_code == 200:
            return len(r.content)
    except Exception:
        pass
    return None

resultados_wp = {}

for m in MUNICIPIOS:
    nombre = m['nombre']
    r_wd = resultados_wd.get(nombre, {})
    if not r_wd.get('ok'):
        continue
    pois_con_articulo = [p for p in r_wd['pois'] if p.get('article')][:3]
    print(f"\n=== {nombre} ({len(pois_con_articulo)} POIs a consultar) ===")
    resultados_wp[nombre] = []
    for p in pois_con_articulo:
        titulo = titulo_desde_url(p['article'])
        res = wiki_summary(titulo)
        if not res['ok']:
            print(f"  ERROR {titulo}: {res['error'][:80]}")
            continue
        data = res['data']
        extract = data.get('extract', '')
        thumb = (data.get('thumbnail') or {}).get('source')
        thumb_w = (data.get('thumbnail') or {}).get('width')
        thumb_h = (data.get('thumbnail') or {}).get('height')
        thumb_bytes = tamaño_miniatura(thumb) if thumb else None
        primeras_frases = '. '.join(extract.split('. ')[:2])
        print(f"  {p['label'][:40]}")
        print(f"    extract: {len(extract)} chars · primeras 2 frases: {len(primeras_frases)} chars")
        print(f"    thumb: {thumb_w}x{thumb_h} · {thumb_bytes/1024:.1f} KB" if thumb_bytes else "    thumb: (sin miniatura)")
        print(f"    → {primeras_frases[:150]}...")
        print(f"    summary call: {res['ms']:.0f} ms, {res['bytes']} B")
        resultados_wp[nombre].append({
            "poi": p['label'],
            "titulo": titulo,
            "extract_len": len(extract),
            "primeras_frases": primeras_frases,
            "thumb_url": thumb,
            "thumb_dims": (thumb_w, thumb_h) if thumb else None,
            "thumb_bytes": thumb_bytes,
            "ms": res['ms'],
            "json_bytes": res['bytes'],
        })
        time.sleep(0.3)

# Resumen: tamaño total estimado por pueblo si solo cargamos 1 thumbnail (la decisión del usuario)
print("\n--- Resumen prueba 2 ---")
print("Tamaño estimado por pueblo asumiendo 1 foto principal + 2 descripciones texto")
total_kb = 0
for nombre, items in resultados_wp.items():
    if not items: continue
    # 1 foto (la del primer POI)
    kb_foto = (items[0]['thumb_bytes'] or 0) / 1024
    # 3 llamadas summary JSON
    kb_json = sum(i['json_bytes'] for i in items) / 1024
    subtotal = kb_foto + kb_json
    total_kb += subtotal
    print(f"  {nombre:32s}  foto: {kb_foto:5.1f} KB · jsons: {kb_json:5.1f} KB · total: {subtotal:5.1f} KB")
print(f"  {'TOTAL vista 3 (4 pueblos)':32s}  {total_kb:5.1f} KB")

## Prueba 3 — Overpass como fallback

**Estrategia:** para los municipios donde Wikidata haya dado pocos POIs (menos de 3, por ejemplo), consultamos Overpass en un radio de 2 km alrededor del centro del pueblo con los tags:

- `historic=*` (cualquier elemento histórico)
- `tourism=viewpoint` (miradores)
- `tourism=attraction` (atracciones turísticas)
- `natural=peak` (picos)
- `building=castle|church|monastery` (edificios concretos)

**Qué medimos:**
- Número de elementos devueltos.
- Cuántos tienen `name` (los que no, no nos sirven).
- Latencia (Overpass puede ser lento).
- Tamaño de la respuesta.

**Qué no nos da Overpass:** descripciones ni fotos. Solo nombre, coordenadas y tipo. Si un pueblo depende de este fallback, en la vista 3 saldrá el POI con nombre pero sin texto ni foto. Habrá que decidir si eso es aceptable o si preferimos ocultar el pueblo.

In [ ]:
def overpass_pois_cercanos(lat, lon, radio_m=2000):
    q = f'''[out:json][timeout:25];
(
  node["historic"](around:{radio_m},{lat},{lon});
  way["historic"](around:{radio_m},{lat},{lon});
  node["tourism"="viewpoint"](around:{radio_m},{lat},{lon});
  node["tourism"="attraction"](around:{radio_m},{lat},{lon});
  way["tourism"="attraction"](around:{radio_m},{lat},{lon});
  node["natural"="peak"](around:{radio_m},{lat},{lon});
  way["building"~"^(castle|church|monastery|cathedral|chapel)$"](around:{radio_m},{lat},{lon});
);
out center tags;
'''
    return fetch_json(OVERPASS, method="POST", data={"data": q})

resultados_ov = {}

for m in MUNICIPIOS:
    nombre = m['nombre']
    r_wd = resultados_wd.get(nombre, {})
    n_wd = len(r_wd.get('pois', []))
    print(f"\n=== {nombre} (Wikidata dio {n_wd} POIs, lanzamos Overpass de todas formas) ===")
    res = overpass_pois_cercanos(m['lat'], m['lon'], radio_m=2000)
    if not res['ok']:
        print(f"  ERROR: {res['error'][:120]}")
        resultados_ov[nombre] = {"ok": False, "error": res['error']}
        continue
    elementos = res['data'].get('elements', [])
    con_nombre = [e for e in elementos if e.get('tags', {}).get('name')]
    print(f"  {len(elementos)} elementos · {len(con_nombre)} con name · {res['ms']:.0f} ms · {res['bytes']} B")
    for e in con_nombre[:10]:
        tags = e.get('tags', {})
        name = tags.get('name', '?')
        tipo_info = []
        for k in ('historic', 'tourism', 'natural', 'building'):
            if k in tags:
                tipo_info.append(f"{k}={tags[k]}")
        tipo = ' '.join(tipo_info)
        print(f"    {name[:40]:40s}  [{tipo}]")
    if len(con_nombre) > 10:
        print(f"    ... y {len(con_nombre)-10} más")
    resultados_ov[nombre] = {
        "ok": True,
        "ms": res['ms'],
        "bytes": res['bytes'],
        "total": len(elementos),
        "con_nombre": len(con_nombre),
        "elementos": con_nombre,
    }
    time.sleep(2)  # Overpass agradece pausas

print("\n--- Comparativa Wikidata vs Overpass ---")
print(f"{'Municipio':32s} {'WD':>6s} {'WD+img':>8s} {'OV':>6s} {'OV+nom':>8s}")
for m in MUNICIPIOS:
    nombre = m['nombre']
    r_wd = resultados_wd.get(nombre, {})
    r_ov = resultados_ov.get(nombre, {})
    n_wd = len(r_wd.get('pois', [])) if r_wd.get('ok') else -1
    n_wd_img = r_wd.get('con_imagen', 0) if r_wd.get('ok') else 0
    n_ov = r_ov.get('total', 0) if r_ov.get('ok') else -1
    n_ov_n = r_ov.get('con_nombre', 0) if r_ov.get('ok') else 0
    print(f"{nombre:32s} {n_wd:>6d} {n_wd_img:>8d} {n_ov:>6d} {n_ov_n:>8d}")

## Prueba 4 — Datos del municipio (bloque 1.5 de regalo)

**Estrategia:** una segunda consulta SPARQL por municipio pidiendo: población (último valor disponible), altitud, superficie, comarca, coordenadas del centro, escudo, bandera, enlace al artículo en Wikipedia en español.

**Qué medimos:**
- Qué campos están disponibles en cada municipio.
- Latencia.
- Si esto cierra el bloque 1.5 del plan.

In [ ]:
def query_municipio(qid):
    return f'''
SELECT ?poblacion ?altitud ?superficie ?comarcaLabel ?articulo WHERE {{
  OPTIONAL {{ wd:{qid} wdt:P1082 ?poblacion. }}
  OPTIONAL {{ wd:{qid} wdt:P2044 ?altitud. }}
  OPTIONAL {{ wd:{qid} wdt:P2046 ?superficie. }}
  OPTIONAL {{ wd:{qid} wdt:P361 ?comarca. }}
  OPTIONAL {{
    ?articulo schema:about wd:{qid} ;
              schema:isPartOf <https://es.wikipedia.org/> .
  }}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "es,en". }}
}}
LIMIT 1
'''

resultados_muni = {}

for m in MUNICIPIOS:
    q = query_municipio(m['qid'])
    res = consultar_wikidata(q)
    if not res['ok']:
        print(f"{m['nombre']:32s}  ERROR: {res['error'][:80]}")
        continue
    rows = res['data']['results']['bindings']
    if not rows:
        print(f"{m['nombre']:32s}  (sin datos)")
        continue
    row = rows[0]
    pob = row.get('poblacion', {}).get('value', '-')
    alt = row.get('altitud', {}).get('value', '-')
    sup = row.get('superficie', {}).get('value', '-')
    com = row.get('comarcaLabel', {}).get('value', '-')
    art = row.get('articulo', {}).get('value', '-')
    print(f"{m['nombre']:32s}  pob={pob:>8s}  alt={alt:>6s}  sup={sup:>6s}  com={com[:25]:25s}  ({res['ms']:.0f} ms)")
    resultados_muni[m['nombre']] = {
        "poblacion": pob, "altitud": alt, "superficie": sup, "comarca": com, "articulo": art, "ms": res['ms']
    }
    time.sleep(1)

print("\nCampos disponibles por municipio:")
for nombre, d in resultados_muni.items():
    campos = [k for k in ('poblacion','altitud','superficie','comarca') if d.get(k) and d[k] != '-']
    print(f"  {nombre:32s} {len(campos)}/4  ({', '.join(campos)})")

## Prueba 5 — Pueblos cercanos a un punto GPS

**Esto es crítico.** La vista 3 del panel muestra 4 pueblos cercanos al punto GPS actual. Necesitamos una forma fiable de obtener esa lista dada una latitud/longitud.

**Estrategia a evaluar:** Overpass con `place=village|town|city` en un radio de 15 km, ordenados luego en cliente por distancia al punto.

**Prueba:** usamos 3 puntos de carretera real (no sobre un pueblo) para simular el caso real del coche circulando:
- Punto en A-2 cerca de Medinaceli.
- Punto en A-5 cerca de Trujillo.
- Punto en la N-VI cerca de Astorga (zona de Castrillo de los Polvazares).

**Qué medimos:**
- Número de pueblos encontrados.
- Latencia de la consulta.
- Coherencia del listado (los primeros 4 deberían ser pueblos relevantes, no pedanías ni lugares sin interés).
- Tamaño de la respuesta.

In [ ]:
import math

def distancia_m(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2-lat1)
    dl = math.radians(lon2-lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(a))

def overpass_pueblos_cercanos(lat, lon, radio_m=15000):
    q = f'''[out:json][timeout:25];
(
  node["place"~"^(village|town|city)$"](around:{radio_m},{lat},{lon});
);
out tags;
'''
    return fetch_json(OVERPASS, method="POST", data={"data": q})

PUNTOS_CARRETERA = [
    {"nombre": "A-2 cerca de Medinaceli",       "lat": 41.1856, "lon": -2.4570},
    {"nombre": "A-5 cerca de Trujillo",         "lat": 39.4760, "lon": -5.8200},
    {"nombre": "N-VI cerca de Astorga",         "lat": 42.4420, "lon": -6.0800},
]

resultados_cercanos = {}

for p in PUNTOS_CARRETERA:
    print(f"\n=== {p['nombre']} ({p['lat']}, {p['lon']}) ===")
    res = overpass_pueblos_cercanos(p['lat'], p['lon'], radio_m=15000)
    if not res['ok']:
        print(f"  ERROR: {res['error'][:120]}")
        continue
    elementos = res['data'].get('elements', [])
    # Calculamos distancia y ordenamos
    pueblos = []
    for e in elementos:
        tags = e.get('tags', {})
        name = tags.get('name')
        if not name: continue
        d = distancia_m(p['lat'], p['lon'], e.get('lat',0), e.get('lon',0))
        pueblos.append({"name": name, "place": tags.get('place'), "dist": d, "pop": tags.get('population')})
    pueblos.sort(key=lambda x: x['dist'])
    print(f"  {len(pueblos)} pueblos con nombre en radio 15 km · {res['ms']:.0f} ms · {res['bytes']} B")
    print("  Primeros 10 por distancia:")
    for pu in pueblos[:10]:
        pop = f"pop={pu['pop']}" if pu['pop'] else ""
        print(f"    {pu['dist']/1000:5.1f} km  {pu['name'][:30]:30s} [{pu['place']}] {pop}")
    resultados_cercanos[p['nombre']] = {"ms": res['ms'], "bytes": res['bytes'], "total": len(pueblos), "top4": pueblos[:4]}
    time.sleep(2)

## Resumen final

Celda que reúne todo en un informe compacto copiable al documento de seguimiento.

In [ ]:
print("="*70)
print("RESUMEN FASE 1.4 — POIs por pueblo")
print("="*70)

print("\n## Wikidata SPARQL (prueba 1)")
print(f"{'Municipio':32s} {'POIs':>6s} {'Img':>6s} {'Wiki':>6s} {'ms':>8s}")
for m in MUNICIPIOS:
    r = resultados_wd.get(m['nombre'], {})
    if r.get('ok'):
        print(f"{m['nombre']:32s} {len(r['pois']):>6d} {r['con_imagen']:>6d} {r['con_articulo']:>6d} {r['ms']:>8.0f}")

print("\n## Wikipedia REST (prueba 2) — tamaño vista 3 (1 foto + 3 jsons por pueblo, 4 pueblos)")
total = 0
for nombre, items in resultados_wp.items():
    if not items: continue
    kb_foto = (items[0]['thumb_bytes'] or 0) / 1024
    kb_json = sum(i['json_bytes'] for i in items) / 1024
    total += kb_foto + kb_json
    print(f"  {nombre:32s}  {kb_foto+kb_json:5.1f} KB")
print(f"  TOTAL estimado vista 3: {total:.1f} KB")

print("\n## Overpass fallback (prueba 3)")
print(f"{'Municipio':32s} {'elem':>6s} {'con nom':>8s} {'ms':>8s}")
for m in MUNICIPIOS:
    r = resultados_ov.get(m['nombre'], {})
    if r.get('ok'):
        print(f"{m['nombre']:32s} {r['total']:>6d} {r['con_nombre']:>8d} {r['ms']:>8.0f}")

print("\n## Datos de municipio / bloque 1.5 (prueba 4)")
for nombre, d in resultados_muni.items():
    campos = [k for k in ('poblacion','altitud','superficie','comarca') if d.get(k) and d[k] != '-']
    print(f"  {nombre:32s} {len(campos)}/4 campos disponibles: {', '.join(campos)}")

print("\n## Pueblos cercanos vía Overpass (prueba 5)")
for nombre, r in resultados_cercanos.items():
    print(f"  {nombre:32s} {r['total']:3d} pueblos en 15 km · {r['ms']:.0f} ms · {r['bytes']} B")
    for pu in r['top4']:
        print(f"    {pu['dist']/1000:5.1f} km  {pu['name']}")

print("\n" + "="*70)
print("Pega este resumen en el chat para que Claude lo analice")
print("y redacte las fichas de API y la decisión arquitectónica.")